# ECOD Architecture Mapping

Maps each protein chain from the CompleteProteins set to its **ECOD architecture classification**.

The ECOD (Evolutionary Classification of protein Domains) database groups protein domains
into a hierarchy: **Architecture → X-group → H-group → Topology → Family**.
This notebook extracts the top-level *architecture* label (e.g. `beta barrels`,
`alpha arrays`, `alpha/beta barrels`) for each of the 160 target chains and saves
the result as `ecod_mapping.csv`.

The ECOD domains file (`ecod.v294.1.domains.txt`) should already be present in this
directory. If not, the notebook will download the latest version from the ECOD server.

A later section merges **mean-pooled SAE embeddings** with ECOD labels and runs **PCA + t-SNE**. Embeddings are read from **`latent_analysis/protein_latent_mean_pooled.npz`** (written by **`latent_umap.ipynb`**) if present, otherwise from **`token_sae_latents/*_latents.npy`** if you export those files yourself.

In [ ]:
import os
import urllib.request
from pathlib import Path

import pandas as pd

In [ ]:
PROTEIN_IDS = [
    '6tf4_A', '6wmk_A', '6wqc_A', '6zpp_A', '7ab3_E', '7ar0_B', '7atr_A', '7au7_A',
    '7awk_A', '7b0d_A', '7b1k_B', '7b1w_F', '7b26_C', '7b28_F', '7b29_A', '7b2a_A',
    '7b2o_A', '7b3a_A', '7b4q_A', '7b7t_A', '7bbz_A', '7bcz_A', '7bew_A', '7bhy_A',
    '7cjs_B', '7dcm_A', '7dfe_A', '7djy_A', '7dk9_A', '7dkk_A', '7dko_A', '7dmf_A',
    '7dms_A', '7dnu_A', '7don_B', '7dq9_A', '7dru_C', '7dtp_A', '7dup_A', '7dut_A',
    '7dvn_A', '7e0m_A', '7e2v_A', '7e37_A', '7e3z_A', '7ebt_B', '7ee3_C', '7ef6_A',
    '7eqx_A', '7esx_A', '7et8_A', '7ewu_B', '7ezg_A', '7f0h_A', '7f6e_B', '7f8a_A',
    '7fbh_A', '7fbp_B', '7fe3_A', '7ffa_A', '7fh3_A', '7fip_A', '7jiz_A', '7jt9_A',
    '7kdx_B', '7kik_A', '7kiu_A', '7kos_A', '7kqv_D', '7ksp_A', '7kua_A', '7kzh_A',
    '7l6j_A', '7l6y_A', '7l8n_A', '7lbu_A', '7lc5_A', '7lgr_A', '7ljh_A', '7lnu_B',
    '7lqm_D', '7lqn_A', '7ls0_B', '7lvz_D', '7m4n_A', '7mcc_A', '7mfi_A', '7mfw_B',
    '7mpz_A', '7mrk_D', '7mro_A', '7mu9_A', '7mwr_A', '7ndr_A', '7nl4_A', '7ntn_A',
    '7ny6_A', '7o49_F', '7obm_A', '7ofn_A', '7on9_A', '7ool_A', '7ouq_A', '7p3b_B',
    '7p3t_B', '7p82_C', '7pab_A', '7pbk_A', '7plb_B', '7poh_B', '7ppp_A', '7pq7_A',
    '7pqf_A', '7prd_A', '7puo_A', '7q03_A', '7q1b_A', '7q47_A', '7r84_A', '7rbw_A',
    '7rdt_A', '7re4_A', '7re6_A', '7s94_C', '7sf6_A', '7sir_A', '7spo_C', '7spp_C',
    '7stt_A', '7stz_C', '7swh_A', '7swk_B', '7sy9_A', '7t24_A', '7t5f_B', '7t71_A',
    '7t9w_B', '7ta9_A', '7tav_B', '7tbs_A', '7tcb_B', '7thw_A', '7tj1_D', '7tkv_A',
    '7v1v_A', '7v5y_B', '7vdy_A', '7vmu_A', '7vnb_A', '7vpf_A', '7vw0_B', '7w83_A',
    '7wbr_A', '7wgk_A',
]

PROTEIN_ID_SET = set(PROTEIN_IDS)
print(f"{len(PROTEIN_IDS)} target protein chains")

In [ ]:
def find_or_download_ecod() -> Path:
    """Return path to a local ECOD domains file, downloading if necessary."""
    here = Path(".").resolve()

    for candidate in sorted(here.glob("ecod*.domains.txt")):
        print(f"Using local ECOD file: {candidate.name}")
        return candidate

    ecod_url = "http://prodata.swmed.edu/ecod/distributions/ecod.latest.domains.txt"
    dest = here / "ecod.latest.domains.txt"
    print(f"No local ECOD file found — downloading from {ecod_url} …")
    urllib.request.urlretrieve(ecod_url, str(dest))
    print(f"Saved to {dest.name}")
    return dest


ecod_path = find_or_download_ecod()
ecod_path

## Parse ECOD and match target proteins

ECOD tab-separated columns (v294+):

| idx | column |
|-----|--------|
| 0 | uid |
| 1 | ecod_domain_id |
| 2 | manual_rep |
| 3 | f_id |
| 4 | pdb |
| 5 | chain |
| 6 | pdb_range |
| 7 | seqid_range |
| 8 | **architecture_name** |
| 9 | x_name |
| 10 | h_name |
| 11 | t_name |
| 12 | f_name |
| … | (flags) |

In [ ]:
COL_PDB = 4
COL_CHAIN = 5
COL_ARCH = 8
COL_X = 9
COL_H = 10
COL_T = 11
COL_F = 12


def parse_ecod(ecod_file: Path) -> pd.DataFrame:
    """Parse the ECOD domains file and return matches for PROTEIN_IDS.

    For chains with multiple ECOD domains we keep only the first hit
    (typically the largest domain).
    """
    seen: set[str] = set()
    rows: list[dict] = []

    with open(ecod_file, "r") as fh:
        for line in fh:
            if line.startswith("#"):
                continue
            parts = line.rstrip("\n").split("\t")
            if len(parts) <= COL_F:
                continue

            pdb_id = parts[COL_PDB].lower()
            chain_id = parts[COL_CHAIN]
            query_id = f"{pdb_id}_{chain_id}"

            if query_id not in PROTEIN_ID_SET or query_id in seen:
                continue
            seen.add(query_id)

            rows.append({
                "protein_id": query_id,
                "architecture": parts[COL_ARCH].strip(),
                "x_group": parts[COL_X].strip(),
                "h_group": parts[COL_H].strip(),
                "t_group": parts[COL_T].strip(),
                "family": parts[COL_F].strip(),
            })

    df = pd.DataFrame(rows)
    print(f"Matched {len(df)} / {len(PROTEIN_IDS)} target proteins in ECOD.")
    unmatched = sorted(PROTEIN_ID_SET - seen)
    if unmatched:
        print(f"Unmatched ({len(unmatched)}): {unmatched}")
    return df


ecod_df = parse_ecod(ecod_path)
ecod_df.head(10)

In [ ]:
print("Architecture distribution:\n")
print(ecod_df["architecture"].value_counts().to_string())

In [ ]:
output_csv = Path("ecod_mapping.csv")
ecod_df.to_csv(output_csv, index=False)
print(f"Saved → {output_csv}  ({len(ecod_df)} rows)")

## Quick sanity check

Re-read the CSV and display a summary.

In [ ]:
check = pd.read_csv(output_csv)
display(check)
print(f"\n{check.shape[0]} proteins, {check['architecture'].nunique()} unique architectures")

## Latent fingerprints and t-SNE vs ECOD

**Preferred:** run **`latent_umap.ipynb`** once; it saves **`latent_analysis/protein_latent_mean_pooled.npz`** with arrays `protein_ids` and `X_latent_mean` (already pooled). This notebook searches that file under `AUTOENCODER_ROOT`, the current directory, and `original_SAE` parent layouts.

**Alternate:** put per-protein `(L, L, d)` tensors as `token_sae_latents/*_latents.npy` (not produced by default training—only reconstructions are).

**Pre-PCA:** `StandardScaler` by default (override with **`ECOD_LATENT_PREP`**: `standard`, `l2`, or `both`). **t-SNE** perplexity defaults to **40** (**`ECOD_TSNE_PERPLEXITY`**; clipped below `n_samples`).

Env overrides: **`TOKEN_SAE_LATENTS_NPZ`**, `TOKEN_SAE_LATENTS_DIR`, `AUTOENCODER_ROOT`, `ECOD_MAPPING_CSV`.

In [ ]:
# --- Imports for latent geometry check (safe to re-run after cells above) ---
import glob
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler, normalize

# Notebook directory (latents and CSV are resolved from here)
SCRIPT_DIR = Path(".").resolve()
AUTOENCODER_ROOT = Path(os.environ.get("AUTOENCODER_ROOT", str(SCRIPT_DIR.parent))).resolve()
# Directory of optional *_latents.npy (L, L, d) tensors if you export them yourself
LATENT_DIR = Path(
    os.environ.get("TOKEN_SAE_LATENTS_DIR", str(SCRIPT_DIR / "token_sae_latents"))
).resolve()
ECOD_CSV_PATH = Path(os.environ.get("ECOD_MAPPING_CSV", str(SCRIPT_DIR / "ecod_mapping.csv"))).resolve()
# ECOD column to color by: "architecture" (broad) or "family" (fine)
HUE_COLUMN = os.environ.get("ECOD_TSNE_HUE", "architecture")
# Before PCA: "standard" (StandardScaler), "l2" (row-wise L2 unit norm), "both" (scaler then L2)
LATENT_PREP = os.environ.get("ECOD_LATENT_PREP", "standard").lower().strip()
# t-SNE perplexity (< n_samples); default 40; set ECOD_TSNE_PERPLEXITY=30 to match older runs
TSNE_PERPLEXITY_TARGET = int(os.environ.get("ECOD_TSNE_PERPLEXITY", "40"))
# Dead-feature filter: drop dims with max(|x|) ≤ this over all proteins (see ECOD_DEAD_FEATURE_MIN_ALIVE)
DEAD_FEATURE_EPS = float(os.environ.get("ECOD_DEAD_FEATURE_EPS", "1e-10"))
DEAD_FEATURE_MIN_ALIVE = int(os.environ.get("ECOD_DEAD_FEATURE_MIN_ALIVE", "2"))
# Output of latent_umap.ipynb (preferred on ICE — training does not write *_latents.npy by default)
LATENT_NPZ_NAME = "protein_latent_mean_pooled.npz"


def latent_npz_search_paths() -> list[Path]:
    """Default locations for protein_latent_mean_pooled.npz (same layout as latent_umap.ipynb)."""
    paths: list[Path] = []
    env_npz = os.environ.get("TOKEN_SAE_LATENTS_NPZ", "").strip()
    if env_npz:
        paths.append(Path(env_npz).expanduser().resolve())
    for root in (
        SCRIPT_DIR,
        SCRIPT_DIR.parent,
        AUTOENCODER_ROOT,
        AUTOENCODER_ROOT / "original_SAE",
        SCRIPT_DIR.parent.parent,
    ):
        paths.append((root / "latent_analysis" / LATENT_NPZ_NAME).resolve())
    seen: set[Path] = set()
    out: list[Path] = []
    for p in paths:
        if p not in seen:
            seen.add(p)
            out.append(p)
    return out


def normalize_protein_id(pid: str) -> str:
    """Align latent filenames with ECOD rows: pdb lower, chain upper."""
    s = str(pid).strip()
    if "_" not in s:
        return s
    pdb, chain = s.split("_", 1)
    return f"{pdb.lower()}_{chain.upper()}"


def load_latents_from_npz(npz_path: Path) -> pd.DataFrame | None:
    """Rows compatible with load_and_pool_latents from latent_umap.npz output."""
    try:
        data = np.load(npz_path, allow_pickle=False)
    except OSError:
        return None
    if "protein_ids" not in data.files or "X_latent_mean" not in data.files:
        return None
    ids = data["protein_ids"]
    X = np.asarray(data["X_latent_mean"], dtype=np.float64)
    if X.ndim != 2 or X.shape[0] != len(ids):
        return None
    rows: list[dict] = []
    for i in range(X.shape[0]):
        raw = str(ids[i])
        row = {"protein_id_raw": raw, "protein_id": normalize_protein_id(raw)}
        for j in range(X.shape[1]):
            row[f"feature_{j}"] = float(X[i, j])
        rows.append(row)
    return pd.DataFrame(rows)


def load_and_pool_latents(latent_dir: Path, glob_pattern: str = "*_latents.npy") -> pd.DataFrame:
    """Load (L, L, d) latent .npy files, global-mean pool to (d,) fingerprints."""
    print("1. Loading and pooling latent tensors…")
    latent_paths = sorted(glob.glob(str(latent_dir / glob_pattern)))
    if not latent_paths:
        print(f"  No files matched {(latent_dir / glob_pattern)!r}")
        return pd.DataFrame()

    rows: list[dict] = []
    for path in latent_paths:
        stem = Path(path).stem  # e.g. 7ar0_B_latents
        protein_id = stem.replace("_latents", "").rstrip("_")
        z = np.load(path)
        if z.ndim != 3:
            raise ValueError(f"{path}: expected 3D (L, L, d), got shape {z.shape!r}")
        pooled = z.mean(axis=(0, 1)).astype(np.float32, copy=False)
        row = {"protein_id_raw": protein_id, "protein_id": normalize_protein_id(protein_id)}
        for i, v in enumerate(pooled):
            row[f"feature_{i}"] = float(v)
        rows.append(row)

    out = pd.DataFrame(rows)
    print(f"  Loaded {len(out)} proteins, d_latent = {len([c for c in out.columns if c.startswith('feature_')])}")
    return out


def load_latents_for_ecod_tsne() -> pd.DataFrame:
    """Prefer NPZ from latent_umap; else optional *_latents.npy directory."""
    for npz_path in latent_npz_search_paths():
        if not npz_path.is_file():
            continue
        df = load_latents_from_npz(npz_path)
        if df is not None and not df.empty:
            d = len([c for c in df.columns if c.startswith("feature_")])
            print(f"1. Using mean-pooled latents from {npz_path}  (n={len(df)}, d={d})")
            return df
    print("  No protein_latent_mean_pooled.npz found; checked:")
    for p in latent_npz_search_paths():
        print(f"    - {p}")
    return load_and_pool_latents(LATENT_DIR)


In [ ]:
# 2–5: merge ECOD labels, PCA, t-SNE, plot
df_latents = load_latents_for_ecod_tsne()
if df_latents.empty:
    print("Skipping t-SNE: run latent_umap.ipynb to write latent_analysis/protein_latent_mean_pooled.npz,")
    print("  or set TOKEN_SAE_LATENTS_NPZ, or add *_latents.npy under TOKEN_SAE_LATENTS_DIR.")
else:
    print("2. Attaching ECOD classifications…")
    df_ecod = pd.read_csv(ECOD_CSV_PATH)
    df_ecod = df_ecod.assign(protein_id=df_ecod["protein_id"].map(normalize_protein_id))

    df_merged = pd.merge(df_latents, df_ecod, on="protein_id", how="inner", suffixes=("", "_ecod"))
    print(f"  Merged {len(df_merged)} / {len(df_latents)} latent rows with ECOD ({len(df_ecod)} CSV rows).")
    if df_merged.empty:
        print("  No overlap: check protein_id naming vs CSV and latent filenames.")
    elif HUE_COLUMN not in df_merged.columns:
        raise KeyError(f"HUE_COLUMN={HUE_COLUMN!r} not in merged columns: {list(df_merged.columns)}")
    elif len(df_merged) < 2:
        print("Need at least 2 merged proteins for PCA/t-SNE.")
    else:
        feature_cols = [c for c in df_merged.columns if c.startswith("feature_")]
        X_raw = np.asarray(df_merged[feature_cols].values, dtype=np.float64)
        n_samples, n_feat_before = X_raw.shape
        # Drop near-dead columns (max |x| ≤ eps over all proteins). Skip filter if that leaves <2 dims.
        alive_mask = np.max(np.abs(X_raw), axis=0) > DEAD_FEATURE_EPS
        n_alive = int(np.sum(alive_mask))
        if n_alive < DEAD_FEATURE_MIN_ALIVE:
            print(
                f"  Dead-feature filter: only {n_alive} dims exceed |x|>{DEAD_FEATURE_EPS:g} — "
                f"skipping (need ≥{DEAD_FEATURE_MIN_ALIVE}) so PCA/t-SNE stay well-posed."
            )
            n_feat = n_feat_before
        else:
            n_dead = n_feat_before - n_alive
            X_raw = X_raw[:, alive_mask]
            n_samples, n_feat = X_raw.shape
            print(
                f"  Dead-feature filter (|x|>{DEAD_FEATURE_EPS:g} in ≥1 protein): "
                f"dropped {n_dead} / {n_feat_before} dims; keeping {n_feat}."
            )

        n_comp = min(50, n_samples - 1, n_feat)
        if n_comp < 1:
            print("Could not choose PCA dimensionality.")
        else:
            if LATENT_PREP == "l2":
                X_for_pca = normalize(X_raw, norm="l2", axis=1)
                print("  Pre-PCA: L2 normalize rows (unit length)")
            elif LATENT_PREP in ("both", "standard+l2"):
                X_for_pca = normalize(
                    StandardScaler().fit_transform(X_raw), norm="l2", axis=1
                )
                print("  Pre-PCA: StandardScaler + L2 row normalization")
            else:
                X_for_pca = StandardScaler().fit_transform(X_raw)
                print("  Pre-PCA: StandardScaler (zero mean, unit variance per feature)")

            print(f"3. PCA: {n_feat} → {n_comp} components…")
            pca = PCA(n_components=n_comp, random_state=42)
            X_pca = pca.fit_transform(X_for_pca)

            perp = min(max(5, TSNE_PERPLEXITY_TARGET), n_samples - 1)
            tsne_init = "pca" if X_pca.shape[1] >= 2 else "random"
            if tsne_init == "random":
                print("  t-SNE init: random (PCA input has <2 dimensions; init='pca' would fail).")
            print(f"4. t-SNE (2D), perplexity={perp} (target {TSNE_PERPLEXITY_TARGET}, capped by n={n_samples})…")
            tsne = TSNE(
                n_components=2,
                perplexity=perp,
                random_state=42,
                init=tsne_init,
                learning_rate="auto",
            )
            X_tsne = tsne.fit_transform(X_pca)
            df_plot = df_merged.copy()
            df_plot["tsne_x"] = X_tsne[:, 0]
            df_plot["tsne_y"] = X_tsne[:, 1]

            print("5. Plot…")
            fig, ax = plt.subplots(figsize=(12, 10))
            hue_series = df_plot[HUE_COLUMN].astype(str)
            cats = sorted(hue_series.unique())
            cmap = plt.get_cmap("tab20")
            for i, cat in enumerate(cats):
                m = hue_series == cat
                ax.scatter(
                    df_plot.loc[m, "tsne_x"],
                    df_plot.loc[m, "tsne_y"],
                    label=cat,
                    alpha=0.85,
                    s=60,
                    edgecolors="none",
                    color=cmap(i % 20),
                )
            ax.set_title("Latent space (mean-pooled SAE) vs ECOD labels", fontsize=16, fontweight="bold")
            ax.set_xlabel("t-SNE 1", fontsize=12)
            ax.set_ylabel("t-SNE 2", fontsize=12)
            leg = HUE_COLUMN.replace("_", " ").title()
            ax.legend(title=leg, bbox_to_anchor=(1.02, 1), loc="upper left")
            plt.tight_layout()
            out_png = SCRIPT_DIR / "ecod_tsne_plot.png"
            plt.savefig(out_png, dpi=300, bbox_inches="tight")
            plt.show()
            print(f"Saved → {out_png}")
